# Module 5 — Audit Trail Generation (Colab version)

**Goal:** plain-English recommendation, per-claim evidence chains, confidence labels, ATCS score, saved report files.

This keeps the memory-safe phase separation from the local version: retrieval models load, do their work, and get freed before the LLM loads. On the local 8GB RAM machine this was the difference between running and freezing; on Colab it matters less, but is kept anyway as sound practice for a long working session where several large models get loaded over time.

**Model choice:** `deepseek-r1:1.5b`, not `7b`. This is the model already reported for the thesis's evaluation figures (Module 5 onward), so it stays fixed here to keep results consistent with what is already written up, rather than switching to `7b` just because Colab has more headroom than the local 8GB machine did.

Two corrections carried over from the local version, same reasoning as Modules 1–4: the hardcoded local model path is replaced with the Hugging Face model name, and query embedding pooling is corrected to attention-masked mean, matching the corpus embeddings.

Requires Modules 1–3's outputs (FAISS index, chunk metadata, populated Neo4j graph) to already exist.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install FAISS on its own

In [2]:
!pip install -q faiss-cpu

## 3. Install and start Ollama, then pull `deepseek-r1:1.5b`

Same install-and-poll sequence as `00_setup.ipynb` and Module 4. If this Colab session already has Ollama running from a prior cell in the same runtime, this cell is safe to re-run — it just confirms the server is already up.

In [3]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess, time, requests

ollama_process = subprocess.Popen(["ollama", "serve"])

for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time — re-run this cell.")

Ollama server is up.


In [5]:
!ollama pull deepseek-r1:1.5b

## 4. Install the remaining Python packages

In [6]:
!pip install -q transformers neo4j python-dotenv ollama
!pip install -q nmslib-metabrainz==2.1.3
!pip install -q --no-deps scispacy
!pip install -q conllu pysbd scikit-learn scipy joblib

In [7]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


## 5. Imports, model config patch, and Neo4j connection

In [8]:
import os
import re
import gc
import glob
import json
from datetime import datetime
import pandas as pd
import numpy as np
import faiss
import torch
import en_core_sci_sm
from neo4j import GraphDatabase
from google.colab import userdata

# --- Patch the scispaCy model config before loading (same fix as Modules 2-4) ---
pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
for path in glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True):
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)

# --- Neo4j Aura connection ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OLLAMA_MODEL = "deepseek-r1:1.5b"  # matches the model already used for the thesis's reported evaluation figures

REPORTS_DIR = f"{PROJECT_ROOT}/reports/audit_trail"
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Setup ready.")

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.15). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Setup ready.


## 6. RETRIEVAL PHASE — load Bio-ClinicalBERT + scispaCy, run search, then free them

Everything needed from the retrieval models is collected here (vector matches + graph concepts with their CUIs). Once this function returns, those models are deleted from memory before moving on.

The pooling fix from Modules 1, 3, and 4 applies here too — the local version's `outputs.last_hidden_state.mean(dim=1)` averaged over padding tokens; this masks them out first.

In [9]:
def run_retrieval_phase(query, k=5):
    from transformers import AutoTokenizer, AutoModel
    from scispacy.linking import EntityLinker

    faiss_index = faiss.read_index(os.path.join(PROCESSED_DIR, "pmc_patients.index"))
    chunks_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    bert_model.eval()

    nlp = en_core_sci_sm.load()
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
    linker = nlp.get_pipe("scispacy_linker")

    # --- Vector search (attention-masked mean pooling) ---
    inputs = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)

    mask = inputs["attention_mask"].unsqueeze(-1)
    summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    query_vec = (summed / counts).cpu().numpy().astype("float32")

    distances, indices = faiss_index.search(query_vec, k)
    vec_matches = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        vec_matches.append({"patient_id": str(row["patient_id"]), "rank": rank + 1, "text": row["text"][:400]})

    # --- Query entity extraction + graph search ---
    doc = nlp(query)
    query_entities = []
    for ent in doc.ents:
        if ent._.kb_ents:
            cui, score = ent._.kb_ents[0]
            name = linker.kb.cui_to_entity[cui].canonical_name
            query_entities.append({"cui": cui, "name": name})

    graph_matches = []
    if query_entities:
        cuis = [e["cui"] for e in query_entities]
        with driver.session() as session:
            result = session.run(
                """
                MATCH (c:Concept)<-[:MENTIONS]-(p:Patient)
                WHERE c.cui IN $cuis
                RETURN p.patient_id AS patient_id, count(DISTINCT c) AS matched_concepts,
                       collect(DISTINCT {name: c.canonical_name, cui: c.cui}) AS concepts
                ORDER BY matched_concepts DESC LIMIT $k
                """,
                cuis=cuis, k=k
            )
            graph_matches = [dict(r) for r in result]

    known_concepts = {}
    for gm in graph_matches:
        for c in gm["concepts"]:
            known_concepts[c["name"].lower()] = c["cui"]

    context = "\n\n".join(f"Patient {m['patient_id']}: {m['text']}" for m in vec_matches)

    # --- Free heavy models before returning ---
    del bert_model, tokenizer, nlp, linker, faiss_index
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "context": context,
        "vec_matches": vec_matches,
        "graph_matches": graph_matches,
        "known_concepts": known_concepts,  # {name_lower: cui}
        "concept_names": list({c["name"] for gm in graph_matches for c in gm["concepts"]})
    }

print("Retrieval function defined (not run yet).")

Retrieval function defined (not run yet).


## 7. REASONING PHASE — Ollama call, generate tagged recommendation

By this point, Bio-ClinicalBERT and scispaCy are already out of memory.

In [10]:
REASONER_PROMPT_TEMPLATE = """You are a clinical reasoning assistant. Using ONLY the patient context below, \
answer the clinician's query. For each factual claim you make, tag it clearly like this: \
[CLAIM: your claim text]. If you cannot support a claim from the context, do not state it.

Patient context:
{context}

Known related concepts from the knowledge graph: {concepts}

Clinician query: {query}

Provide a short clinical recommendation with tagged claims."""

def run_reasoning_phase(query, retrieval_result):
    import ollama

    prompt = REASONER_PROMPT_TEMPLATE.format(
        context=retrieval_result["context"] or "No context retrieved.",
        concepts=", ".join(retrieval_result["concept_names"]) or "None",
        query=query
    )
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]
    claims = re.findall(r"\[CLAIM:\s*(.*?)\]", raw_text)
    return raw_text, claims

print("Reasoning function defined (not run yet).")

Reasoning function defined (not run yet).


## 8. VERIFICATION PHASE — uses only data already collected

Checks each claim against the concept names/CUIs already gathered during retrieval — no need to reload scispaCy.

In [11]:
def verify_claim(claim_text, retrieval_result):
    claim_lower = claim_text.lower()
    known_concepts = retrieval_result["known_concepts"]  # {name_lower: cui}

    for name_lower, cui in known_concepts.items():
        if name_lower in claim_lower:
            patient_id = None
            for gm in retrieval_result["graph_matches"]:
                if any(c["cui"] == cui for c in gm["concepts"]):
                    patient_id = gm["patient_id"]
                    break
            evidence = f"[UMLS: {name_lower} ({cui}) -> MENTIONS -> Patient {patient_id}]"
            return {"text": claim_text, "status": "Verified", "evidence_chain": evidence}

    if any(w.lower() in retrieval_result["context"].lower() for w in claim_text.split() if len(w) > 5):
        return {
            "text": claim_text,
            "status": "Partially Verified",
            "evidence_chain": "Text overlap with retrieved context, but no confirmed graph path."
        }

    return {
        "text": claim_text,
        "status": "Unverifiable",
        "evidence_chain": "No supporting evidence found in either pathway."
    }

print("Verification function defined (not run yet).")

Verification function defined (not run yet).


## 9. Auditor — build report + ATCS score + save to Drive

In [12]:
def compute_atcs(verified_claims):
    total = len(verified_claims)
    if total == 0:
        return 0.0
    verified = sum(1 for c in verified_claims if c["status"] == "Verified")
    return round((verified / total) * 100, 1)

def build_audit_report(query, raw_recommendation, verified_claims):
    atcs = compute_atcs(verified_claims)
    return {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "recommendation": raw_recommendation,
        "claims": verified_claims,
        "atcs_score": atcs,
        "total_claims": len(verified_claims),
        "verified_claims": sum(1 for c in verified_claims if c["status"] == "Verified"),
        "partially_verified_claims": sum(1 for c in verified_claims if c["status"] == "Partially Verified"),
        "unverifiable_claims": sum(1 for c in verified_claims if c["status"] == "Unverifiable"),
    }

def render_human_readable(report):
    lines = [
        "=" * 60, "CLINICAL AUDIT REPORT", "=" * 60,
        f"Generated: {report['timestamp']}",
        f"Query: {report['query']}", "",
        "Recommendation:", report["recommendation"], "",
        "-" * 60, "Claim-by-claim verification:", "-" * 60,
    ]
    for c in report["claims"]:
        lines.append(f"\n[{c['status']}] {c['text']}")
        lines.append(f"  Evidence: {c['evidence_chain']}")
    lines += [
        "", "-" * 60,
        f"Audit Trail Coverage Score (ATCS): {report['atcs_score']}%",
        f"  Verified: {report['verified_claims']} / {report['total_claims']}",
        f"  Partially Verified: {report['partially_verified_claims']} / {report['total_claims']}",
        f"  Unverifiable: {report['unverifiable_claims']} / {report['total_claims']}",
        "=" * 60,
    ]
    return "\n".join(lines)

def save_report(report, human_readable_text):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    json_path = os.path.join(REPORTS_DIR, f"audit_report_{ts}.json")
    txt_path = os.path.join(REPORTS_DIR, f"audit_report_{ts}.txt")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(human_readable_text)
    print(f"Saved: {json_path}")
    print(f"Saved: {txt_path}")

print("Auditor functions defined (not run yet).")

Auditor functions defined (not run yet).


## 10. Full pipeline — run in sequence, memory-safe

In [13]:
def run_audited_query(query):
    print("[1/4] Running retrieval (loads Bio-ClinicalBERT + scispaCy)...")
    retrieval_result = run_retrieval_phase(query)
    print(f"      -> {len(retrieval_result['vec_matches'])} vector matches, "
          f"{len(retrieval_result['graph_matches'])} graph matches. Models freed.")

    print("[2/4] Running reasoning (calls Ollama)...")
    raw_recommendation, claim_texts = run_reasoning_phase(query, retrieval_result)
    print(f"      -> {len(claim_texts)} claims generated.")

    print("[3/4] Verifying claims against graph data already retrieved...")
    verified_claims = [verify_claim(c, retrieval_result) for c in claim_texts]

    print("[4/4] Building and saving report...")
    report = build_audit_report(query, raw_recommendation, verified_claims)
    human_readable = render_human_readable(report)
    save_report(report, human_readable)

    print("\n" + human_readable)
    return report

In [14]:
query = "What should be considered for a patient presenting with COVID-19 and respiratory distress?"
report = run_audited_query(query)

[1/4] Running retrieval (loads Bio-ClinicalBERT + scispaCy)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the curr

https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpa48u8608


100%|██████████| 492M/492M [00:17<00:00, 30.4MiB/s]


...

100%|██████████| 628M/628M [00:21<00:00, 30.3MiB/s]


Finished download, copying /tmp/tmp9driiv__ to cache at /root/.scispacy/datasets/d5e593bc2d8adeee7754be423cd64f5d331ebf26272074a2575616be55697632.0660f30a60ad00fffd8bbf084a18eb3f462fd192ac5563bf50940fc32a850a3c.umls_2022_ab_cat0129.jsonl
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/umls_semantic_type_tree.tsv not found in cache, downloading to /tmp/tmp22zf106f


100%|██████████| 4.26k/4.26k [00:00<00:00, 8.73MiB/s]

Finished download, copying /tmp/tmp22zf106f to cache at /root/.scispacy/datasets/21a1012c532c3a431d60895c509f5b4d45b0f8966c4178b892190a302b21836f.330707f4efe774134872b9f77f0e3208c1d30f50800b3b39a6b8ec21d9adf1b7.umls_semantic_type_tree.tsv


      -> 5 vector matches, 5 graph matches. Models freed.
[2/4] Running reasoning (calls Ollama)...
      -> 0 claims generated.
[3/4] Verifying claims against graph data already retrieved...
[4/4] Building and saving report...
Saved: /content/drive/MyDrive/ClinicalTrust/reports/audit_trail/audit_report_20260827_041735.json
Saved: /content/drive/MyDrive/ClinicalTrust/reports/audit_trail/audit_report_20260827_041735.txt

CLINICAL AUDIT REPORT
Generated: 2026-08-27T04:17:35.150227
Query: What should be considered for a patient presenting with COVID-19 and respiratory distress?

Recommendation:
The patient presenting with COVID-19 and respiratory distress should be considered in the context of the risk of COVID-19 causing respiratory distress, especially in severe cases. While COVID-19 does not always cause respiratory distress, it is a significant factor that should be noted. Other factors, such as genetic predisposition or underlying conditions, could also influence the patient's respir

## Next steps

1. Run `run_audited_query(...)` on a few more queries, choosing ones that match concepts actually present in the current Module 2 subset. Each call reloads/frees BERT and scispaCy independently, so runs are slower per-query but keep peak memory low.
2. Saved reports land in `ClinicalTrust/reports/audit_trail/` on Drive, as both `.json` (for downstream aggregation in Module 7) and `.txt` (for quick reading).
3. Once several runs look correct, Module 6 introduces the baseline systems (LLM-only, standard RAG) that this KG-RAG pipeline gets compared against.
4. Save this notebook into `ClinicalTrust/notebooks/` on Drive alongside Modules 1–4.